In [45]:
import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType , TimestampType 

In [47]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [48]:
sc = spark.sparkContext

In [49]:
%%sh
head /home/talentum/shared/ecomorders_sample.csv 
 wc -l /home/talentum/shared/ecomorders_sample.csv

order_id,order_timestamp,customer_id,city,category,product_id,quantity,unit_price,discount_pct,payment_mode,order_status
ORD000301,2025-09-05 22:01:27,CUST0025,Delhi,Books,P0210,2,201.26,5.0,COD,DELIVERED
ORD000379,2025-12-02 07:06:18,CUST0020,Pune,Fashion,P0290,2,2527.91,0.0,Wallet,DELIVERED
ORD000585,2025-01-19 15:22:26,CUST0089,Ahmedabad,Electronics,P0267,1,10806.18,5.0,COD,DELIVERED
ORD000354,2025-01-24 20:53:43,CUST0023,Pune,Grocery,P0003,1,916.83,25.0,Wallet,DELIVERED
ORD000002,2025-02-17 06:14:32,CUST0011,Mumbai,Fashion,P0230,4,3655.96,5.0,Wallet,CANCELLED
ORD000434,2025-04-04 04:18:07,CUST0096,Ahmedabad,Grocery,P0010,2,964.31,15.0,Credit Card,DELIVERED
ORD000145,2025-01-23 06:56:02,CUST0067,Hyderabad,Beauty,P0019,5,3465.19,10.0,COD,DELIVERED
ORD000545,2025-01-08 15:59:12,CUST0083,Kolkata,Home & Kitchen,P0112,1,4631.69,20.0,Wallet,DELIVERED
ORD000351,2025-01-28 07:54:29,CUST0071,Hyderabad,Grocery,P0050,5,595.88,0.0,COD,CANCELLED
647 /home/talentum/shared/ecomorders_sample.csv


In [50]:


# 1. Load the file using an explicit schema (StructType). 

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

filePath = 'file:///home/talentum/shared/ecomorders_sample.csv'

# Initialize Spark Session
spark = SparkSession.builder.appName("Ecommerce-Orders").getOrCreate()

# Define the explicit schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_timestamp", TimestampType(), True),
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", DoubleType(), True),  # Can be null for missing values
    StructField("payment_mode", StringType(), True),
    StructField("order_status", StringType(), True)
])

# Load the CSV file with the defined schema
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .schema(schema) \
    .csv(filePath)

# df = df.filter(df['discount_pct'].isNull())

# Display the schema and first few rows to verify
df.printSchema()
df.show(5, truncate=False)

# Notes - 
# Null Handling: Columns like discount_pct may have missing values (e.g., blank in CSV), so they are 
# defined as DoubleType() with True for nullable — this treats empty cells as null.


# 1. Remove duplicate order_id rows, drop rows where customer_id is null and replace null discount_pct with 0. 
# Print the record count before and after each cleaning step.

from pyspark.sql.functions import col

print("=" * 55)
# Step 1: Print record count before any cleaning
print("Record count before cleaning:", df.count())

# Step 2: Remove duplicate order_id rows (keep first occurrence)
df_no_duplicates = df.dropDuplicates(["order_id"])
print("Record count after removing duplicates:", df_no_duplicates.count())

# Step 3: Drop rows where customer_id is null
df_no_null_customer = df_no_duplicates.filter(col("customer_id").isNotNull())
print("Record count after dropping null customer_id:", df_no_null_customer.count())

# Step 4: Replace null values in discount_pct with 0.0
df_cleaned = df_no_null_customer.fillna({"discount_pct": 0.0})
print("Record count after replacing null discount_pct with 0:", df_cleaned.count())
print("=" * 55)

# Optional: Show final cleaned data
df_cleaned.show(5, truncate=False)

# Note -
# This solution ensures data integrity and prepares the dataset for downstream analysis like revenue 
# calculation or customer behavior modeling.
# Timestamp Parsing: order_timestamp is parsed correctly as TimestampType assuming the format is 
# yyyy-MM-dd HH:mm:ss.


root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)

+---------+-------------------+-----------+---------+-----------+----------+--------+----------+------------+------------+------------+
|order_id |order_timestamp    |customer_id|city     |category   |product_id|quantity|unit_price|discount_pct|payment_mode|order_status|
+---------+-------------------+-----------+---------+-----------+----------+--------+----------+------------+------------+------------+
|ORD000301|2025-09-05 22:01:27|CUST0025   |Delhi    |Books      |P0210     |2       |201.26    |5.0         |COD        

In [76]:
# Note -
# This solution ensures data integrity and prepares the dataset for downstream analysis like revenue 
# calculation or customer behavior modeling.

# 2.a - Using only DELIVERED orders, compute the total net_amount per category per month (yyyy-MM). 
# net_amount = quantity × unit_price × (1 − discount_pct / 100)

from pyspark.sql.functions import col, to_date, date_format, expr, sum as spark_sum

# Step 1: Filter only DELIVERED orders
delivered_orders = df_cleaned.filter(col("order_status") == "DELIVERED")

# Step 2: Compute net_amount using the formula:
# net_amount = quantity × unit_price × (1 − discount_pct / 100)
# Use expr for clean and readable expression
delivered_with_net = delivered_orders.withColumn(
    "net_amount",
    expr("quantity * unit_price * (1 - discount_pct / 100)")
)

# Step 3: Extract month (yyyy-MM) from order_timestamp
delivered_with_month = delivered_with_net.withColumn(
    "month",
    date_format(col("order_timestamp"), "yyyy-MM")
)

# Step 4: Group by category and month, compute total net_amount
result = delivered_with_month.groupBy("category", "month") \
    .agg(spark_sum("net_amount").alias("total_net_amount")) \
    .orderBy("category", "month")

# Print the result
result.show(truncate=False)
result.printSchema()

# Note -
# Date Extraction: date_format(order_timestamp, "yyyy-MM") extracts the year-month string for grouping.

+--------+-------+------------------+
|category|month  |total_net_amount  |
+--------+-------+------------------+
|Beauty  |2025-01|62145.6585        |
|Beauty  |2025-02|3240.512          |
|Beauty  |2025-03|5129.75           |
|Beauty  |2025-04|49008.74          |
|Beauty  |2025-05|11729.268499999998|
|Beauty  |2025-06|8445.1575         |
|Beauty  |2025-07|12282.703000000001|
|Beauty  |2025-08|23580.850000000002|
|Beauty  |2025-09|15058.882500000002|
|Beauty  |2025-10|14500.4355        |
|Beauty  |2025-11|15463.017         |
|Beauty  |2025-12|30784.756         |
|Books   |2025-01|15510.6675        |
|Books   |2025-02|3388.185          |
|Books   |2025-03|6374.106          |
|Books   |2025-04|4123.794999999999 |
|Books   |2025-05|4119.827          |
|Books   |2025-06|1698.664          |
|Books   |2025-07|7629.8135         |
|Books   |2025-08|2510.49           |
+--------+-------+------------------+
only showing top 20 rows

root
 |-- category: string (nullable = true)
 |-- month: strin

In [77]:

# 2.b - Using a window function, display the top-earning category of every month

from pyspark.sql.functions import col, date_format, expr, sum as spark_sum, dense_rank
from pyspark.sql.window import Window

# Step 1: Filter only DELIVERED orders
df_delivered = df_cleaned.filter(col("order_status") == "DELIVERED")
print("-----1-----")
df_delivered.printSchema()

# Step 2: Create month column in yyyy-MM format
df_delivered = df_delivered.withColumn(
    "month",
    date_format(col("order_timestamp"), "yyyy-MM")
)

print("-----2-----")
df_delivered.printSchema()


# Step 3: Calculate net_amount
df_delivered = df_delivered.withColumn(
    "net_amount",
    expr("quantity * unit_price * (1 - discount_pct / 100)")
)

print("-----3-----")
df_delivered.printSchema()

# Step 4: Calculate total net_amount per month per category
df_month_category = df_delivered.groupBy("month", "category") \
    .agg(
        spark_sum("net_amount").alias("total_net_amount")
    )
print("-----4-----")
df_month_category.printSchema()

# Step 5: Define window specification
# Partition by month and order by total_net_amount descending
window_spec = Window.partitionBy("month").orderBy(col("total_net_amount").desc())

# Step 6: Rank categories within each month
df_ranked = df_month_category.withColumn(
    "rank",
    dense_rank().over(window_spec)
)
print("-----5-----")
df_ranked.printSchema()

# Step 7: Keep only top-earning category of each month
df_top_category_per_month = df_ranked.filter(col("rank") == 1) \
    .select("month", "category", "total_net_amount") \
    .orderBy("month")
print("-----6-----")
df_top_category_per_month.printSchema()

# Step 8: Show final result
df_top_category_per_month.show(truncate=False)

# Note -
# dense_rank() is used here.
# If two categories have the same highest total_net_amount in a month, both will be shown.
# If you want only one category per month, even in case of ties, use row_number() instead of dense_rank().
# Date Extraction: date_format(order_timestamp, "yyyy-MM") extracts the year-month string for grouping.



-----1-----
root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = false)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)

-----2-----
root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = false)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- month: string (nullable = 

In [79]:
# 3. For each city, find the top 3 customers by total net_amount (use dense_rank). Show city, customer_id, 
# total_spend and rank. 

from pyspark.sql.functions import col, expr, sum as spark_sum, dense_rank, round as spark_round
from pyspark.sql.window import Window

# Step 1: Keep only DELIVERED orders (remove this line to include all statuses)
df_delivered = df_cleaned.filter(col("order_status") == "DELIVERED")

# Step 2: Calculate net_amount using the business formula
df_net = df_delivered.withColumn(
    "net_amount",
    expr("quantity * unit_price * (1 - discount_pct / 100)")
)

# Step 3: Total spend per customer within each city
df_city_customer = df_net.groupBy("city", "customer_id") \
    .agg(spark_round(spark_sum("net_amount"), 2).alias("total_spend"))



# Step 4: Window - one group per city, highest spender first
window_spec = Window.partitionBy("city").orderBy(col("total_spend").desc())

# Step 5: Give a rank to every customer inside each city
df_ranked = df_city_customer.withColumn("rank", dense_rank().over(window_spec))

# Step 6: Keep only the top 3 ranks per city
df_top3 = df_ranked.filter(col("rank") <= 3) \
    .select("city", "customer_id", "total_spend", "rank") \
    .orderBy("city", "rank", "customer_id")

# Step 7: Show the result
df_top3.show(100, truncate=False)

# Note - 
# dense_rank() and ties: Customers with the same total_spend get the same rank, and the next rank is not 
# skipped (1, 1, 2, 3 and not 1, 1, 3, 4). Because of this, a city can show more than 3 rows if there are 
# ties.







+---------+-----------+-----------+----+
|city     |customer_id|total_spend|rank|
+---------+-----------+-----------+----+
|Ahmedabad|CUST0091   |157195.63  |1   |
|Ahmedabad|CUST0092   |87389.9    |2   |
|Ahmedabad|CUST0088   |65344.38   |3   |
|Bengaluru|CUST0048   |394271.1   |1   |
|Bengaluru|CUST0044   |294720.22  |2   |
|Bengaluru|CUST0041   |231025.15  |3   |
|Chennai  |CUST0058   |284705.27  |1   |
|Chennai  |CUST0056   |252441.22  |2   |
|Chennai  |CUST0051   |87823.87   |3   |
|Delhi    |CUST0028   |264211.26  |1   |
|Delhi    |CUST0026   |143590.54  |2   |
|Delhi    |CUST0031   |101889.98  |3   |
|Hyderabad|CUST0061   |319263.27  |1   |
|Hyderabad|CUST0062   |136897.1   |2   |
|Hyderabad|CUST0066   |69861.57   |3   |
|Kolkata  |CUST0084   |300443.15  |1   |
|Kolkata  |CUST0083   |231414.33  |2   |
|Kolkata  |CUST0081   |128112.09  |3   |
|Mumbai   |CUST0003   |326304.02  |1   |
|Mumbai   |CUST0008   |59490.38   |2   |
|Mumbai   |CUST0006   |35874.09   |3   |
|Pune     |CUST0

In [80]:
delivered_orders

order_id,order_timestamp,customer_id,city,category,product_id,quantity,unit_price,discount_pct,payment_mode,order_status
ORD000055,2025-12-17 19:51:07,CUST0007,Mumbai,Toys,P0007,1,2615.32,0.0,Net Banking,DELIVERED
ORD000163,2025-12-19 01:10:28,CUST0076,Kolkata,Grocery,P0368,1,563.07,15.0,Debit Card,DELIVERED
ORD000332,2025-06-09 04:06:24,CUST0049,Chennai,Home & Kitchen,P0373,2,2964.54,25.0,Wallet,DELIVERED
ORD000354,2025-01-24 20:53:43,CUST0023,Pune,Grocery,P0003,1,916.83,25.0,Wallet,DELIVERED
ORD000196,2025-11-25 23:41:26,CUST0067,Hyderabad,Books,P0061,1,475.65,15.0,Credit Card,DELIVERED
ORD000313,2025-01-11 10:52:50,CUST0056,Chennai,Beauty,P0069,1,3046.35,5.0,Debit Card,DELIVERED
ORD000574,2025-01-15 13:02:02,CUST0091,Ahmedabad,Electronics,P0232,5,23120.38,0.0,Net Banking,DELIVERED
ORD000372,2025-02-07 01:11:50,CUST0052,Chennai,Sports,P0231,4,11003.05,0.0,COD,DELIVERED
ORD000519,2025-08-20 17:18:24,CUST0095,Ahmedabad,Fashion,P0223,2,4448.41,10.0,COD,DELIVERED
ORD000228,2025-12-22 06:30:20,CUST0029,Delhi,Books,P0167,5,834.87,5.0,Net Banking,DELIVERED


In [120]:

# Step 1: Keep only DELIVERED orders
df_delivered = df_cleaned.filter(col("order_status") == "DELIVERED")

# Step 2: Calculate net_amount using the business formula
df_net = df_delivered.withColumn(
    "net_amount",
    expr("quantity * unit_price * (1 - discount_pct / 100)")
)

# Step 3: Extract the date part from the timestamp
df_net = df_net.withColumn("order_date", to_date(col("order_timestamp")))

# Step 4: Daily revenue = total net_amount per day
df_daily = df_net.groupBy("order_date") \
    .agg(spark_round(spark_sum("net_amount"), 2).alias("daily_revenue"))

# Step 5: Window frame = current row and the 6 rows before it (7 rows in total)
# Ordered by date so the frame moves forward day by day
# No partitionBy is used because we want one continuous series across all dates
window_7d = Window.orderBy("order_date").rowsBetween(-6, 0)

# Step 6: Add the 7-day moving average column
df_result = df_daily.withColumn(
    "moving_avg_7d",
    spark_round(avg("daily_revenue").over(window_7d), 2)
).orderBy("order_date")

# Step 7: Show the result
df_result.show(20, truncate=False)

# Step 8: Save to HDFS in Parquet format
# df_result.write \
#     .mode("overwrite") \
#     .parquet("/user/exam/setA/daily_revenue")
df_result.write \
    .mode("overwrite") \
    .parquet("/user/UseCase1-Spark/daily_revenue")

# Step 9: Read back to verify the saved data
# spark.read.parquet("/user/exam/setA/daily_revenue").show(5, truncate=False)
spark.read.parquet("/user/UseCase1-Spark/daily_revenue").show(5, truncate=False)

# '''
# rowsBetween(-6, 0) counts rows, not calendar days: The frame covers the current row and the 6 rows before it. 
# This equals 7 calendar days only if there is a row for every date. If some dates have no DELIVERED orders, 
# those dates are missing and the average will span more than 7 days.

# Early dates: For the first 6 days, the window has fewer than 7 rows, so the average is calculated on 
# whatever rows are available (for example, day 1 average is just day 1 revenue).

# No partitionBy: Spark will warn that all data is moved to a single partition. This is fine here because the 
# daily summary is small (at most 365 rows for one year).

# mode("overwrite"): This replaces the output folder if it already exists, so the code can be re-run safely. 
# Use "error" or "append" if your exam requires different behavior.

# Absolute HDFS path: /user/exam/setA/daily_revenue starts from the HDFS root. 
# If you want a full URI, use hdfs://<namenode-host>:<port>/user/exam/setA/daily_revenue.
# '''




+----------+-------------+-------------+
|order_date|daily_revenue|moving_avg_7d|
+----------+-------------+-------------+
|2025-01-01|73149.99     |73149.99     |
|2025-01-02|23234.05     |48192.02     |
|2025-01-03|32778.61     |43054.22     |
|2025-01-04|32867.71     |40507.59     |
|2025-01-05|75166.54     |47439.38     |
|2025-01-07|2459.84      |39942.79     |
|2025-01-08|67484.35     |43877.3      |
|2025-01-09|36884.71     |38696.54     |
|2025-01-10|56833.11     |43496.41     |
|2025-01-11|89889.41     |51655.1      |
|2025-01-12|63128.26     |55978.03     |
|2025-01-14|7936.8       |46373.78     |
|2025-01-15|142307.38    |66352.0      |
|2025-01-16|18087.83     |59295.36     |
|2025-01-17|93736.97     |67417.11     |
|2025-01-18|20085.18     |62167.4      |
|2025-01-19|42219.61     |55357.43     |
|2025-01-20|5542.34      |47130.87     |
|2025-01-21|3029.72      |46429.86     |
|2025-01-23|17555.22     |28608.12     |
+----------+-------------+-------------+
only showing top